# Two-Phase Training — LoRA SFT → DPO

**Model:** `Qwen/Qwen2.5-0.5B-Instruct`  
**Device:** Apple MPS (Mac M-series) / CPU fallback  
**Libraries:** HuggingFace `transformers`, `peft`, `trl`

## Pipeline Overview
```
Phase 1 — SFT with LoRA
  └── Load pre-trained Qwen2.5-0.5B-Instruct
  └── Attach LoRA adapters (rank=8)
  └── Train on SFT dataset with SFTTrainer
  └── Save SFT model → models/sft_model/

Phase 2 — DPO
  └── Load SFT model (now the reference + policy model)
  └── Attach fresh LoRA adapters for DPO
  └── Train on DPO dataset with DPOTrainer
  └── Save final model → models/dpo_model/
  └── Push to HuggingFace Hub
```

> **Prerequisites:** Run `data_preparation.ipynb` first.

# PHASE 1 Training - LoRA SFT

## Imports & Device Setup

In [1]:
import os
import gc
import torch
from pathlib import Path
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig

# Device
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"Device       : {DEVICE}")
print(f"PyTorch      : {torch.__version__}")

Device       : mps
PyTorch      : 2.11.0


In [2]:
DATA_DIR   = Path("../data")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

SFT_DATASET_PATH = DATA_DIR / "sft_dataset"
TEST_DATASET_PATH = DATA_DIR / "test_dataset"
SFT_MODEL_PATH   = MODELS_DIR / "sft_model"

## Loading Dataset

In [3]:
sft_data  = load_from_disk(str(SFT_DATASET_PATH))
test_data = load_from_disk(str(TEST_DATASET_PATH))

print(f"SFT   train : {len(sft_data['train']):,}")
print(f"SFT   val   : {len(sft_data['validation']):,}")
print(f"Test        : {len(test_data):,}")
print(f"\nSFT columns  : {sft_data['train'].column_names}")
print(f"Test columns : {test_data.column_names}")

SFT   train : 3,645
SFT   val   : 405
Test        : 810

SFT columns  : ['flags', 'instruction', 'category', 'intent', 'response', 'text']
Test columns : ['flags', 'instruction', 'category', 'intent', 'response']


## Load Tokenizer & Base Model

### Tokenizer and Chat Template for TEST dataset

In [4]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [6]:
SYSTEM_PROMPT ="""You are a knowledgeable and empathetic customer support assistant.
Your role is to help customers resolve their issues quickly and accurately.
Always be polite, clear, and solution-focused."""

In [14]:
# ── Cell 4: Format test set ───────────────────────────────────
def format_test_sample(example):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    example["text"] = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return example

test_formatted = test_data.map(format_test_sample, num_proc=4)
test_for_eval  = test_formatted.select_columns(["text"])

print(f"Test set formatted : {len(test_formatted):,} samples")
print(f"Eval-ready subset  : {len(test_for_eval):,} samples, columns: {test_for_eval.column_names}")

Test set formatted : 810 samples
Eval-ready subset  : 810 samples, columns: ['text']


In [8]:
dtype = torch.float16 if DEVICE == "cuda" else torch.bfloat16

print(f"Loading {MODEL_ID}...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    trust_remote_code=True,
)

model = model.to(DEVICE)
model.config.use_cache = False  # required during training

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Base model loaded — {total_params:.1f}M parameters")

Loading Qwen/Qwen2.5-0.5B-Instruct...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Base model loaded — 494.0M parameters


## Attach LoRA Adapters

In [9]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",              # don't adapt bias terms
    inference_mode=False,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nLoRA adapters attached for SFT")

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184

LoRA adapters attached for SFT


## SFT Training

In [10]:
sft_args = SFTConfig(
    output_dir=str(SFT_MODEL_PATH),

    # Schedule
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,         # effective batch = 16

    # Optimizer
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="adamw_torch",
    weight_decay=0.01,

    # Sequence
    max_length=256,
    dataset_text_field="text",
    packing=False,

    # Evaluation — run on both val split and test set
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=50,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # MPS safe settings
    gradient_checkpointing=False,
    fp16=False,
    bf16=True,
    dataloader_pin_memory=False,

    report_to="none",
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=sft_data["train"],
    eval_dataset=sft_data["validation"],
    processing_class=tokenizer,
)

print("SFTTrainer ready")
print(f"   Train samples    : {len(sft_data['train']):,}")
print(f"   Val samples      : {len(sft_data['validation']):,}")
print(f"   Test samples     : {len(test_formatted):,}")
print(f"   Effective batch  : {2 * 8}")
print(f"   Steps per epoch  : ~{len(sft_data['train']) // (2 * 8)}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


SFTTrainer ready
   Train samples    : 3,645
   Val samples      : 405
   Test samples     : 810
   Effective batch  : 16
   Steps per epoch  : ~227


In [11]:
print("Starting Phase 1 — SFT Training...")
print("=" * 55)

sft_result = sft_trainer.train()

print("=" * 55)
print(f"SFT Training complete")
print(f"   Training loss : {sft_result.training_loss:.4f}")
print(f"   Runtime       : {sft_result.metrics.get('train_runtime', 0)/60:.1f}min")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting Phase 1 — SFT Training...


Step,Training Loss,Validation Loss
50,1.560769,1.093521
100,1.038799,0.969750
150,0.968893,0.921288
200,0.947429,0.905848
228,0.947429,0.905115


SFT Training complete
   Training loss : 1.1074
   Runtime       : 35.2min


## Evaluating on Test Cases

In [19]:
from torch.utils.data import DataLoader

print("Tokenizing test set for evaluation...")

def tokenize_for_eval(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

test_tokenized = test_for_eval.map(
    tokenize_for_eval,
    remove_columns=["text"],
    desc="Tokenizing test set"
)
test_tokenized.set_format("torch")

model.eval()
dataloader = DataLoader(test_tokenized, batch_size=2)

total_loss = 0
total_steps = 0

print("Evaluating on test set...")
with torch.no_grad():
    for batch in dataloader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        total_loss  += outputs.loss.item()
        total_steps += 1

avg_loss = total_loss / total_steps
perplexity = torch.exp(torch.tensor(avg_loss)).item()

print("\n── Test Evaluation Results ──────────────────────────")
print(f"   test_loss       : {avg_loss:.4f}")
print(f"   test_perplexity : {perplexity:.4f}")

Tokenizing test set for evaluation...
Evaluating on test set...

── Test Evaluation Results ──────────────────────────
   test_loss       : 5.5462
   test_perplexity : 256.2665


## Inference check

In [20]:
def generate_response(model, tokenizer, user_query, max_new_token=256):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_query},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(DEVICE)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_token,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    generated = outputs[0][inputs["input_ids"].shape[1]:]  # skip prompt tokens

    return tokenizer.decode(generated, skip_special_tokens=True)

In [21]:
test_queries = [
    "I want to cancel my subscription. How do I do that?",
    "I haven't received my order and it's been 10 days.",
    "How do I reset my account password?",
]

print("── SFT Model Outputs ────────────────────────────────")
for q in test_queries:
    print(f"\nUser  : {q}")
    print(f"Model : {generate_response(model, tokenizer, q)}")
    print("-" * 55)

── SFT Model Outputs ────────────────────────────────

User  : I want to cancel my subscription. How do I do that?
Model : I'm glad you reached out for assistance with cancelling your subscription! Let's get started together. To cancel your subscription, please visit our website and navigate to the account settings page. There, you'll find an option labeled "Subscription Cancellation" or similar. Once there, select "Cancel Subscription" or something similar. This will take you directly to the cancellation process. During this time, we encourage you to contact our customer support team if you have any further questions or concerns about cancelling your subscription. They're available at {{Customer Support Phone Number}} or through {{Website Contact Email}}. Please remember that cancellations can sometimes require a fee, so it's important to review all the details carefully before making a decision. If you need more assistance or have any additional questions during this process, feel fr

## Saving The Model

In [24]:
sft_trainer.save_model(str(SFT_MODEL_PATH/"sft_final"))
tokenizer.save_pretrained(str(SFT_MODEL_PATH/"sft_final"))

print(f"SFT model and tokenizer saved to {SFT_MODEL_PATH}/sft_final")

SFT model and tokenizer saved to ../models/sft_model/sft_final
